# Diffusion Policy on Push-T

Modified from the notebook provided in the
[Diffusion Policy README](https://github.com/real-stanford/diffusion_policy/blob/main/README.md)
(Chi et al., *Diffusion Policy: Visuomotor Policy Learning via Action Diffusion*, RSS 2023).
We use the authors' demonstrations, network and pretrained checkpoint; nothing here trains.
The first run downloads them into `assets/` (about 300 MB).

## Setup

Run these setup cells once per session. The handout notebook includes the helpers in a collapsed setup cell, so it works on its own in Colab or Jupyter. No repository checkout or separate Python files are needed.


In [ ]:
#@title Install notebook dependencies (run once)
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy", "matplotlib", "pymunk", "shapely", "torch",
    "diffusers", "zarr", "gdown", "einops",
])


In [ ]:
#@title Load notebook helpers (run once)
def _load_notebook_helpers():
    import base64, json, sys, types, zlib
    from pathlib import Path
    sources = json.loads(zlib.decompress(base64.b64decode('eJzVXQtz2ziS/is4p65CZWjGdl4T5bR18iOJL36drSQza6kYSoIkrilSQ1K2NXP57/d140GQkieZvdxVnbM7lkigAfS7Gw34j61lGSdFsEiysozT6VZbbD36IT/99NEjcZCNpZhkuUizYTZeieGyFBercpalosxELqMxNfsxw+HfFv2bRyWtJomHvriLy5mIxDAa3ch0LEazrJCpKDJRzqJSxKUolvltfCsLkS/TFOtXPQ7z6EYGBPJ4vsjyUnxZJOUXMcmzuZjJXIo8KvGboKTqaTVom7oJ/PDzOnZFrMABGrX6PFsBRFyISZxIIe/joiwCoUZ/XIhRNJd5BCylY5lT72mGiZazPFtOZ+JT74MvIiwKuJxHo/OrfopHoiijvCzEcgHcj7IIYLJUii9nV93FIolHURln6RcMUk0YTcZyEi2T0uKJFif7KWbWu1GD9G5EVBTLOU9AWmzVwAq0jzF2dpeKL70P7oD9tFgORwlAtLn/JM6LUpTxXBIRsoVMC5DpLk7H2R2TBbMoFJGy4T/kCOj6ss2r/CTzgiD6hABMPXLGn0UFGA00Hc3EXILLxmryNOIiz0aywFJjLCECvGoaRNxgEk+XufRaX5gDDA2fPHkiejKfx2nE9MNoYryUxLzLdBSBDiXoNpILHv/x2dVxehsl8bibT4GrtDwy7x77CiBYvsjStni8fV3Hnbu4QRvQcznKpmn8uxwDGwlQACkCWoC1TIGKU9A6HUkRBMFjmnCPeRqrS0U0JDZDixjyd84YBJdvH/jATwlMaxm0U/dJJoC6Ml89VQ/7WF4KFiyBSxCE0Ij30TxbYgrZBG8gBnEqsmXOBB+RpEe3WTwmJhDb24x1w1CgDDCeiaHEGsoywaKGEnpBOtygKEALGOfRXQoefS9peZGYZ+NlwkhX8hMo4VESHI3KZZQkKwh3lE5loUh+RxwMED2AJ76biVRKTA0tSYKkpDUwzn3i8X5KnVjwfKUeMI0yAyNJYil8BfFlUkgt3dviJE6X9zzUZ+baAmu8lYSjGl2bksaommUx1kXUGxe+IeYoWY5ZRUCCWBXQlwnmSziktUcAnm7HaQmlwOS02MViUyWzwJ6Ch3mM42KRRCvqSgiFHgjU1LuOnCuOIaaAhINo/1n6PFAph1l2gwnHaRKndihoVHSRt9B9kRqoKJdj4kqSvVGWKiKOCRRxYkIqHqRJZJQTxrUkDhM5Z9yhTSInUDxSz+304mS/e/Dh6OyQGIUYjLrI9DbOs5QkisFCYQGdwEMpDTY1NHB1RhPQaMVMU6sgQYuune/dLKMRmFmBIaxeIXBKz2jMUU5cQ+t+w3jSdCr0GmhMLAi95zJSlsOYlGhVMIeep+BKpg1xHa8eLYAKsjfLws5L2RzMYKz0O8Yo8Gg0a9PMMFw/JfUDgpNxG2XFfX/LJUgMESUKF6RdEq1FC8YIQyTqxultlgAC6XEsBRPyBRQKkWli2WAeFwVWAg7gyUVAVh4vFiAnms5AyYQ06HAZJ0xgTK6/1Z1O+1vANgucFRWl0QGjGEHWUp6LYg0tJ0oX3OVxWRKF4BYotcu6uEALtIYy1hJMJswYizlklFjA0N1SdB6TNiYNs8xBn0LCtxiufEMSiHVgXASijjbFWWE/FqvCeVGJLbsWsMETEWJQeBChRr7XEtt/E2eYXVtzx4SAwNxHJZTbXPxLBxgaRzlI0t/SbZQVKJd5avtkRaA5PAD7ef2tSgr6W60H+j0SV9GEERkVN228KkBgwj3xZxoRsnghRU0Ra1VKfpjhDQNuBXlTtI2YTSolZphMM39RyoXimbtsSc2HSifr1fS3yhswqOI6B420tgpzQZLdydzbtDrtPoHnYILG8ZikHJC8ivt9w3hud9gu5xv9OGMvC+lZaK2qmTJ2wtrpBgRoNOjhpayexhNnVp2OmUijH/gvBSFrz+inv3UGz6GIoACFq8mrmRpKtVmkmZ4Z+4ckTskqEMS962BZbGbZHVjyLoa2GWdGFt9gPClcsXpM3BEs0unjVtCE1tpEjn66xvgbBSVYrOgTqQ+MRzyVZr9FbXH0fGePoYTQO2EoOuKaZ9zfGvRTEHNrsSxmJYUeSjxJ5C/waLsnSmZuek9cuV3MIlJHwySDTYxTtorTDEpvgdkZbx/+MTid+sicFfFpNo4nMfqxS15W0Gt2RZmagyyJhtYAkrG6hQ8F2wI4s7JcFO2nT6cYZzkMRtn8KWncbbLkYNfxUwwzWZIHFy4yCM/qKSY6fDqP4vTp5VH38PQomEMpeTDxMpmIaBwtSjMrBBroAV21L6EgYzD/QZKlrI3NsLFuMhwFegZx9hQkTAlfi9mqiEcFySg8U3aFxtqVgZJcwt3Li8ewm/MMbmPOYq1MxAJ0zjFDTERNGtYBor8iKsYUjnzW1mFKWhi/yfqvlJpXDjqZuXEu77Tft1hNwV++OIcNOPjkK097uppD2wOwVEYHYROcxVx5YZZ/mFjHPThXI9gPSV8PssUqZ83ujVpib2fvGZFoOR/GkejmJeg6ikH/YwhTksRT5TFiWZcIOEtCyEnECvyCvPii0GEKGavhSkzzCP2Ap4n2CIG4fIq5E2OlK4HZwltHCFICQez/QyMsgCE25mSns0l5R54wDYnoJsNkiKTjbMT+v/Ltlfh6hIf+1pXuAuXF44zBQZAlxX3mJbMyWUWoddjgkdLClZNoXifxPNaDcFhGiFKeNoTe58n6pPPjCf2WvLbFcpjExcwnBxGwYQrI56WHjHT2n59ysJFgYgBBEZP23cz8dFyVEYYwA40ocpnJsZrX10I+6QSqBINK7gTlVGQ8JgUm2smBwk9gE2h10LrjmDm0reIaSTYGanJkmSEl4kod3YEQi4q8+hVUBfTgUGqssQiTI1YtKacZkOyWMWuQXAlFY6nMlL33R+Lq/G3vc/fySBxfiYvL80/Hh0eHpP6v8IBM0ufj3vvzjz2BNpfds96v4vyt6J79Kj4cnx364uiXi8ujqytxftlPj2Hfj4/w8Pjs4OTj4fHZO7GPjmfn4Pxj8D/g9s4FjalhHR9dEbTTo8uD9/ja3T8+Oe79CtF6e9w7I6hvzy/h2150L3vHBx9Pupfi4uPlxfnVESZwCLhnx2dvLzHM0enRWS/AsHgmjj7hi7h63z05obH6afcjFnBJUxQH5xe/Xh6/e98T789PDo/wcP8Ic+vunxypsbCug5Pu8akvDrun3XdH3OscYLA8aqcmKD6/P6JnNGIX/zvoHZ+f0UoOzs96l/iK4Ov8smf7fj6+OvJF9/L4inDy9vL8FGsknKLLOUNBx7MjBYbwLWqEQRP6/vHqyEIUh0fdEwC7os68StN6kzPIHiCr40VUzsga6xcXkcoJPBLdscon5BImJ0ZQvoK9gc2DpuVcD3kHzHXcUY4xDLuEABCQ+oPxhNR5BNALQ9IMYdgKlPcm4RYtwHJpWVzvDlqukU2X8wWr5HRhny1W82V6U02ebCM8hKnM5lDp3LqY6vU4ShbRylzna1THtzAYXfNwvf0IcilHSjZ0DwS/8sA+3tgF5sa0PoMrHCXx73K94YLyC9K2PIhzhIq+uMiS1TRjz+NHZQf/z5ORjzhyucvym2IRjaSKW1/s7ol7/u8wuxfeMqVon3TkIr6XidU9GZRcjCBN5/5kTlYeHjrpMsogAWMI8VKopzEHa6TZiJzRVL4RhFt6BDNdErQ4F6vt6J4TGoT70QzAPp9ffri66B4cwRnDdPrpu/PuScgqowMeC6I8j1be9d6Ll8GOL/QvPF/E4ql4PmiRc3fvC1gVjFBGUB7voE3Cy+7h8ccrgNh9AUk7Pg3f/52+7OzAkJPIn5+YJ/30Q3jhiw/hJ9UAg+wQUI6u2XV7XIiLQ/bAc2K1HAA/HhxA24UH55+OLjEgeu4Er1848VkpZUjrhgEovGIUJbLzDKATmU7LWee5CRcqN7OHUcq7jMhBqSMgkijC7DHJ4ca0dS4gK4phlPNqUxMEzQOjQAgmXrNja/1nb1sNK54IngkQt+erjy3fafbPt9qptdg4nG0yMOkaOW9O80/Hdd/WB6g1+8utai8HJjlKEYdgRNNEK8KyRxrCT00LCqy9X8LuPjyZcP+gTtJF2D3A8ug1v1XkY01NCg/itcgQNogD9kCJwKLri2lMaSL7ZD8wMSjDMSkfijAAQLepAYhIthH1p+ABzxGL1hvSNwoWweiZKANqIYHo/hJ+7umMbTYsZH7LKhgsmUSUHuc8JaxBDndFZ0J6Zm8BI1M6kRWFmeI8Ho8TaZSIYVpqBh8vF3JM6oH4+17pg3yZ6nRhhFhjWuulEmpoq3UH6ZuK8UHLnlnGY9NGgWIEu+05A9RUUwE0CBEICmlF5pJiC+7pC2/Pb1FySoGbww/HszPoh1bgklp9dpCt6a6ej8BBSpONssJTxGAFVsSp/qoaXhKJXZV3ja7bxcAX19AHo8GgVePNqh2NDD34Ez8q1ENaEfzqcrWQnUmSRWVL/DuPEPQqXiYdtVAGrvCIA76plYA6xvS2YjhScPJe8JD0tlS8SaBqGklP+XpdeD7D8QLsFltDskSgTF13tgbVhEdwvXPQzOOwO6TBfI66+WPHWo76Ot5ybkMJHUeAFKbTwmDuIhU5cmKKQ3keoqjrU37RgQsTnC6TMtY+gXeNB+bzQq1gwWLlorWaacsQkMf/J8HZxbYa7EAvAk7kFMoXUiO3Al7kU/WePits8g4YJx96R+ltDV0qI8F6jzOuQE9lMa1C6rEAUuqJd/4qncFOBCLeWaVtr0GytAzBp+oD5ESh5d58sE+idJrIgVbGHMqmQlOv0n6xDTM322cxznkntczIL2nIqvrGNjpEKF2GoUc5EF+tpkM5VHyWcswf3bQeNSP/WJYet1XNWi5I/fb74KkZXdE2qcq8fOEuXzgfHglIyRgPSR9xwE4w4J2NbgqVBG3VmNROMMQMlgvPSZ9RLpgpBQw6KWLzo9zDRE6j0Upc8qBX3PxGykXBw9KGyhweckzZQs5t0YTGUQkH0ORbzU+utZ2afuAA9BgLGmduFzU7R/Wt5xWv1x+pwXgcSnG+gHf1/EXNEfkft2Rv8PnOD2yaei34I3v4v3Jht9XvDZ0GD6ZDmcxAfFgorJb1RLJWCNzKkUuvxqhO94pZa8yuaXLN1sMrlEJiy8mvBo35sGQHVjg7qtV1e89p+EhcSVV9wKlAEnWErNSuoLTKUunhkSQtpvJxRcGZw7hUFQe8PRs0RmbFEShoZtjnzempRmvT22uvteQYKSDl5+0GO9CcKnioYc9FK3Va3zt4iJevn2zAli+ebJikv768f4VLYlin5Uy8TlqauqKq0pzrOqeX007uF/X6C1OW1IzWoJTCirPxG7MUD8v1re2FV0F6bE35jEsKnxyMVa+0Au/oPEHwSY72xt4TPbuqHc0jJA7LKWns6ajt6VPH/rQayisajWQiVRYZAyCSA348PeD2JtZs4THFek/cl7cSWI7LVUMvrTcQP3XqYz7Bwjf0crhoXDpLHGGS1ou5/j6yD75HuH1yNEfib6IZmtYsnrINTZatZlzR6Iq+emtah5c1zaNbwgWCXsR1G5uMo/mCBL3jvr+LkqSox3z043kvfPFi5yXWwJ9aTR2qGvDrnZcPNFAvbBNA2wxFD7Op0WZNEI3BqtcGK3JKyXSvRmNQYBRSoA6Bgx9D8QGzMn0hbuZlD6yM1jmrwvg+IHgEJmSv3XkafDg+Ozrt9o4PmvRY07re3ouXyg49uJiqo2+GVtmu2hs3i9Kqzd0GxRiu4apXjUh1kz5wlb/3K4UQag8454IO6ggP/A69Ecgd8+bOnfHHGztEy0KOA9er0b07bE31QuYZkScE9tll9pQFMRPsYNwmXoxz7xJB9dIDuB0or8kMrFuTu+5VYHxaWMtide0tIa212TIpoxdmk7ASrobpUKNf7ww2NP7JvN3d8NaxE5Tx+JZlVCz07JsspBf1RI3c6v/oJOmh2nX+kZlQDhAPzk/OLwXtlz96vSPl653+FnHnCe3mvKMMQT/dPzk/+OA2fPXq559fv3YaXiVwH97Bvpt8o9P4+e7L13JXNb7MVlGynyxlFcSyDg4jxNNedG/UcHTPHt19Es89KFWbFG3VXq/otX1HOTUul+AURzOvUesYFQtEhV5/S/62jJL+Vh3sfUlRhXc9aIzmPDaTp1IA7ThG99ptVCF4p5cvaTcuWcyiDvyAegxOxLTBt+/E27YqkiM55WQS0Jp/gQiG+jkma3N0vJYAMD9YEfg25CS/Z4NscmqSLO9UbOGL37N8LPPObqtVFX+sD2S8xxcDd6TaKPUJOGNOIEVqXIfNfM6JqcdgoRfRy1evntFensIn/9dObs+RzIpk1dBGnRsPvK7Nzaodvt08zPNWk/C0d1Qw4fkTIM2jRYcqKKJiHtF0qUTvLh6XM7DACwN2J/gZXuNSafJaGOwwh4xGM+H12HpqZqBB9Gyp1GKlKq84VB5HOZciDnkTNqhyEnrDB7oqFV/MmF/YKS44sAYYU2TMfIcxdDWkShfXGI+pjxbEALxkh9r8vFNLuuFJTWmys6DjYVBjdONdU5vr9vbuQOHwerc9wEdKWoLnqr6ER+pY399qWgQ9QMPRYaLQfxrPeZIdDkzYv8Z6PQOiRaZhHt17BmWEqloDX+w2Pao0y+cdu6dGemutScUP9lNzVg7n1d9oNnzmr3G7w/HVnqDHONMse3HSPXN18vjl3qu9n1X+RzE0Ve+FdgfSM5sSFIxCP+VRbDQaJ9VuoeAQ4u/UoxFfpBi/6Dz3OedeMG/30zp3d92aKjsg7/BwlSQxtU/hbsTs/6IFuv625PKRbGJmpACqMpUzncicq2x0Tzm+XGAN5ubfZAfENI/HNl/H82uLjGvLqErZdoZCeN+UusLW6sIZY8nkhK+eBpe1Ryodd2WiOzNp3oDg8h04voXgci0uWIE3N60L1zePJVQkqQuaetayxkF9D9JxPKdKuL1mHqMwAX9xDSjgQ5I3xxFj5KDNGRcWTfR3nS7jauuamNPb1jqkVOVMq8ECdo2qHAgzC97O49Tjz+AfvYg0p0LZjtj2tlPiMX5tTFBMhbD3vAqunlsOeWfV0QbcXXOjTx0KCGTH2wteULpAPdVfeKCfxE7wnJgGNJNo+TZKCu0xOHYv4nQ8jRxM4O9cp+1Bw9ph8XAqssmEXYoq0GRu8cpWw2LHYDp2G1oEWKbLOVdOe7/HC88Oo70KqKR1I87l226cwSSuOVT1d01/5bocNFq45KZK0Q0Z0s3qZnOe708VtvvDg14DI+XAWORKabmGlK2nMaTPH0gv0o/Wly8eaNJaf7yOClZ8yikTfxM7GxDRcASui+s22v4kdn0BRh84hl4Jw7Zw7RpYE/wLJVTCQQHXUPz7RylcpdoOdidfReFwFHUqyc6HSbTKlqVhAJ0MqVWNeCwtSlupX0VHG3+ryM2HllPzGdqKyar689A8IveNFBp90rsjNgtuKxqd8sVUllRu8UDJ5/9FYefBLBayBNsEVKHcXEhbfIqLZTbP6CiPXtsJJIsrC2+pnlGX09iO5NtdXl1x2WMLsfpnycW8qqijHqn7+qlGgvBMAZIuulFvq6ol2M44TVU9Hm3zTrjeBCufx2XLHptClwrB/XQ0k6Mb3qZ9Y0v9ue6fW5ANcmjQLK5y6ojsswmYqMygKO2TOHuwNF9/XuYJlQ7lZPmK0j6GNqNqKr1c2iXh3baquMg++n9U4oXBRzOzJOYK2iYtQAcq468+ghDheLyYm8UcHl6cXplWfVOhxSJnmmgl6bhV6oHZndRfK8XjPlARqHpQd+mMs2KtAx609Ph/6nW8hxydUC1jR0BAE2/tUIRpABXVUoQ6T0X38lQcXHwshEeHriT8MIhVloKDL1Y9Qh6ijLMzxO0fKM9+myVLXfmcqyAEFmAaj6gC95HaDFcFaW1RRFRtjVk+21v3tUrSILs7P4vCaiR1nml3FyoUoHR9Lh+cSPiQSsxbCQxVsouGlkLGfIz0LloFVLSL6Qa6wh5uVbrAR04JwD5PC4+9hdb/65q47tXVUY8o/A3ZgGXqb5HwloVSIWq31Cr9d1lG2DykDWdVXd2GjtwRs+UcotlUjCbZoXUTQTMmQ21ozxkDCDQEJH40o9Nbh91eF3PlfKA2VaPRLA5vX4VQCzCIwe9wSwMoHXUaZffDr7vH6ckyX5ze/sfh5XD3+cnrs+SXXjg5Kg5uP338jQ3rwfujgw8X58dnNcAsTCGGl4sA6rXUEOfvD/Pw8OjvV+P4XfZ6JUcnL3b29//z198vf377j1kSMsRH4rttlq+cQeh6o6Q5+asOzRIkPlCaJHw4hXiPt52g9O6AY6nOgVLF+zKnwms6RTqJ74FDZTOAs48XVz1Yw1M4U6dUxU8h4IthtPMqGr18+XJ3PHzx6lk0efn6xfD57uvo1evnr+Ro+Grn+cvXOxOXzKBx/DtXk1dEY5ExVCPpUpMvpD6QzAVT5/tX4fvzy+O/n58RLGenxh51GMcjLOLi8ujQtNTSXXAwqM458X6oU3f4SMh7OVpSRKNKmG1ffYg1Km442JpifnDeSLSgA/8rw7+H3Eb+cWfYphwxdxP/FTX+NbrpGZtZjdviZ9118Y1/tqtGBfXdfdlPHcxRah+RvIuhDrdpLL2jBuWzJ3CqH+t9YqMsVSpluahctiwZw167a2buAifNM7Ljj1QlPxV8qZOVhZ1tEktTOTdZwvukovzT7i/h2pRq0952GYJ8ZbXOw+NTKnq1C1Lfseazj6f48vbtxyt6fNyjsnddvGrzcdldmmTR2GPl1Ch4Yhmh0FVXPFl1pc5/qONKrNSeCi+DIdF+lnaoacHsQ7jResoeNVfWhPEY02EAOrmgUmFaqT7ltjY4p6CKocUFq9naSTtteae0HHeTSYEK5jfjOPf4EHKY3XBCxgkmuFdgMRGPO2Z6PhUaLpZlh9wfTsv54rdlLMuOsV1O/LDQ/pZCrJpRuFygq4zmXqtRHFU2zjE1dJvRV1qhW9eCU5msvHhPg9dO8q8snNZlihY1+tTr6Mj5c/AMZdUcf/uPhvL76iTRiRQeAyGz1uwKNU40Ioy7JIKXu74Z1d/665FKBK8C5HnaPOmHVaxNWtmzjYk/daIRnQ414Unh/YFZfrX7GvTD0l730AN8pfPAHn63yL+FtV9Aqpvxvp4oVq1d+eDv8eItsW6cBfsraN/jc8/0Dcg18Vqt2gwlDBK6fxd91hHlLvy7BEFPOJD3JdU6wnQ2yKWgNBID16nytzjJoyGQ5CYYxGsxvwTqEg/CpadW1VIXMATkGfLj/lawINYZrKdpOTOXM8O1zClcE5+4KToTssDsy7ykPLLt5uxAEKnDukflrRes1hu0SclzopNSpNogqBJepfHkIi7geIbs6Na2nHQsB9+qykSShe3ws4C+UPUwOOOKnntWDWmHrcXH3WSnv5VbvtTyywCYERnkpoZKM/1RIam/hZX0t9oM45rOcpdRf2uAT+y00ce2SwJ4rSqG39TFvGr0oTiGHK6irDyZGLi/JzvCWWCNrsAdx0WhM9pcmtHqDZwxvzrmTG3JgL638agyEZSR4mBktBxHpJyi2yhmv3DDeW3dVIPob1Efi1ELyYY180XxVyGiS5NEzTEXS5246qf/7sT5ut5WZV+0QZVlW3dP0+BUHYfX506seTkwBxCj5CM67B5qVjQRdbsZYCtGjUqwPrlVBPAPzTvw51yuwOM53wagNoZ8vtPh3n7/+tWkc2lp7dpC3VRvavd/VM3ZvS9u5Gq95uw0WigeGsdzmRZOPTY706TECV/w3nh3CvYTRvB6e9cXu4P1SlfKsXNNAi31GgMO1mjn3VMB2LVe5YA2uDz+SoscNN5RTeY2+WXVwpbpt5f2nfOAx8elF1Sb9uAUfnK+NbSesgueQv6GDVR2fav0ozUowsRb6vITIFSH/ZQEqSFVgcZqavysfrEp0A14L6QprAYEvGdA2KinVRPlpNtAik6kkU2weKYQDq4D3XEk9f1MDeYwcBYy364YaR6riz3mlJGeR1y27nJPJRWUbq4oBHq5csArYCIGtEXDm7I7LUcw3AbRvWnwtb7Jgdf6kgiWOt8RupbVe5WJcTxNZ1us6RMEZCKSwD4ORpViCJekGcaNdBr9rGkP3+7xuARh41/KEcUz1d04lp2qXGtgNRfwuAbcGThO4XzDk5x3qtDGMTTTJBti3rQGbmSCoSdunOS7dMpTmYS8qeVubzAAHUCG6pAd79nWdrIwt4B5UmU3qK0zU8XvbLitBa+SIy1mpzDJ1PUjHa3gfXEn+YB7SFcOuN6YM2aZGfkJ5G2UGPJa1Q0U1nS3u523nIeM/ZA2h3n7orMhJqyZbj6QoWGxMBSq1vobyYsKxBAGOzQQsNLit2UE1I6yIhwBB7d7fLuORX0SL0KlTVyc048mCDEvVzKSB1DECW8i1LCkFaSyih6m6FfYUbuAha8VjzGpNk0fjGBLZN150GqyrhtdBQpy0ldOt0GZ0RUQheSz+puOK9V6/vDyNtbZ+pqLHwXWnpMiiQ2V1gl1Esurf3U8LPLMd8W/dUS9BT1ZT224PlIUwxh8ipKlPMrzLG/EHBOj9yzA+bLga5HAaXd0F9Mu8+kf64N8fbN2uQygTTHPP+ogvzZiROYR7VNlVP/IF8Rw1R2zqsGJTaB7xkJCV4ezuFAbUlOZ0sZ0lm+wtVfKgq5l4u32ns4DDFfgIzpLJitlbqsxnNHawtv3a3pPaHXYMo7Z5rSiWXYtuUgGiL0sqPF9dt5pF4gS2Zgnnd70TfbLPUth1wvfT2vEd+aRsG6E9gpU9jYHYFhkytCmGfhAQ7pk0aH9N1qVmwTzRWUKWmaLidbHpS8m86lyawqWcycOvELP8cfGLddR2HAYkw5SOjjWhRg7g01uSFxU13zYRDxdHoKlWY6gwoRSprxLC8+Cy3wUqFt1ISLpEkLxOj21VaNlG/tJQWnHYLXyMp0aE2fy7ICwK6HVJs3WempREWJeBaQv5WMa6sinesdHd57tGR3aqZGxDox+BXqRHgf9bJh3rePmHFNbkJvwLlpCcUapoj7d/Kha3tuZqeNOFZ/+KUO4Mmc/bZ54zYQ8opHVBXNSzSQuxab0KRvSti1n0jeHOVsxCppNzPM7BTAlx99s3VTmO4czSKcMi2xuLuDzIroHb7i0N5mRpFjppTEBaEyXP4okpssn6bofuhGSB6LI1LCIXrEdjXffrDPgbVihU7dzw+WKTQi2t6PC1cAVK9LhRWXTsWTToXPjux5bh/7jJJ/uq/7OdOnkC0NHzMSR0wb6tgKg+1Z7EfWUrILnhmD3AVwvrxWw5vBaDbfaWD7aKKEKEReL9mLLmK7Xqe0Yk6OluxSdXbuNfB8qn2tvZ0c/07KojnOajWU57pj3dbPUqZszv1kSeLlMXVMxSrICmiXJsoWSMC5DU9M1JdbuTBVlzMgFXytcjma+3obhDaC65TXbFyaPQ/skesvI4J0ecaDl7VLstG6U37iZCT6O0diDalnzZk2bKTcU3mYDV7NddIkT6M/XI9sNBPpxLmhl1VS3YVh2aq+iUjl3jToHuuPn7tsCYP+h1tq6YYWeU5seqUuwWg1T55miUXOuq9U2VYcwLZ5LOL/iLa6YUoAMNtR0akepS3XHmVYdvHi+jNX4GnruVYAr37CmMlNZn4EeZUjos62YJ3QTww/kXo8DuqhUv9DngacZ89eEzmjytdWjEZ0Q9u1d1vaKUbo2g7mBr9mccc2BtFXWazb7e5xW6xjwKRxTIOLVRLvVPEfort8cvwGWdwCCINGJmsY5WVYXE8PGG85NmzeYBEEa1CNXVwhidWDIKTmCGYOj4FnXAiwBnHSc/s4KaIKDhpNmLaz10byNBj6YR+mSsAL6eM6xa1sSe60nv2k0yxtcePu7zLOijkgDSxW86lN+qshEPfy3itkdzD0SF8rEbtz6rji76e/WVIUjzaZj50EPvx5HGH+/uj7D1ubPFN652N8Sb+C6Jms7HGpFOv2mg15z8TnfZMLV2H+y52ydXQVJRxWqDOaOL2QxKXgXF9tiF5J5pDbdG7p+HVyFVdprsBf2ZdkNX+rIFwnp6avCAfeENbt9nebo9YxXWcma1s+q209ic8zp9o1Jfd/Wy4GJDZtttdDyMddQSa5yMvQaVSXthhpXS8nreGCK4gBoQ0tdIf6tZkYy0JBquaN7z3mC+a1dbsDavnYWkn7iiRGezkZBMT9DqOKbmnNkWVfXSVeW54cnKY7u6Sw4HwD58TeSFSOZ2mqKtVrbJJtmbfdo2D8oe2A3p+g82GOucKtdiaVPlfGdP/piSNpmp6u9ITG9TF3gWjVcLlRAx3L4yDmdpgelmiS8nlItrbrNdShnMV9IRLeHkqN2RyGscvOzUl30enL+7jy86nV7jTvMXvOhqL3d13yVGX7t4tfrZ40bzXT/Rfj5Qk2iIyqA6jQBf+c7m2rv9toDe8pwlt05x1uMj1IdU2nUNdD97LQard99fRaFj0wV5ISQK+ocMXHOnrT11ezGzXaubffVV/3nFFgNRXQPU7woaxm3+o3JdD1zvFCVEfbq3v2d3Z8J5VicLphLVubvChyrMoqn/7FEL7MLpi9UPoPTwekpt+pkIkwNZ0PitIid2QCffgiNLOq1E0N1lKpDIU4S1F6DvLYhRCCcVL+efGBuyddp/Pe90xPjT//VCdDgHFR41DoIJ/G0nnIl4OpdmYX/KGblPPFa/ws5Tsl/PODHJziZuxt7TDAfnd2Hjp3ai4LMab6y+msP8m5juWZjf+ybm1v8h0I6eo9obdfbul65ORCYh9c7HLsUfB5Qtwg3H/XZ4wiQTgbtrZ3v8cybFt+NUOUB+MQLhqD1rh+xySkeoeHrV0FsPkvTOEdjFslpqcF1MWg2dQ6INNu2pTkp4hwWrZxuykZvEE5XniwXKI+POd8mco1UkLOqGhS+idX/PEyvsw5fNs3bnU6krm7254CGbkGp8kl8fE/rTtrCtuG04ybX5qQzPw+nkW3wQX804E55bzbAVotsGx1s4+kq0GQ4JhvSUrExWUJz9lWDd5WcvsC4jqNmsKsGrCH3waReuxnAWm+3QoJeLRVeAmt0d4GDvA1R4jePA37zFN4Pi6PMqUEbH3lm7O9FkCOrjg+9Y6E87D7r+mTNIxWd65ns9je2DuqVyc6hJqUZ2R8wmRB9KwcrDidAiMf3+nbDBBqmOlxcrhe9+uqcGLQUVsguyKaAmjQjj6nJet32aZBBA6d67/2vBIJGQ6xv89TsNh3La9Nc1bm2Og3I5dcj0VkB4qxGgAke2P1O+q+Fkyb7SXNoGDuzGgXT5DeNi9ipe4w6sUQOYse6ivrhre1DjGbvK3Mm3Hm210xxbrCqri5b02LqxdjeLKV3RmrJtMYSfHVfKZfN8V8PAcPdOi9v8fZ21dKbtsTnVVK+Z8GpGyVrN59SpBypvxlirhKorhDwEGa3qosEFBCvlppQ2iSivzpTZTh7jUyd/oNf9poovr73jWs6+IAA12ST46r/dpq2etDLD6U6wE2105EiWpAACnGV1Rwb7YNEeRLTGW59nUthXB0jX3w5I1XAQT1MJvq2VQfN6m/fECiF15lMxsBAnCQ1d0ivqbO29e4oaXPpaTrCF9qo864dYhOhjNdw60Q6jlp3puVeZGoTeGFNaagcTiVSjUumbK5wPcJ+cJZQYTfwr6rp1c/+u0uwgrwplWh+7N6M3hgee0pXuqk/p8ugttP1w8zWX8mYOdkyfX+Fg3fKejpaYD1R1nJc2qZDa33YF47Tusn73HTjjPuq8jT1wv4pV1JFVfJ/oFdrW0nP17aSft7ZtG1Enbho2+reH76htGHjyKlV+KYydlXsJu2qWqg/72a8OzucmzCwqrOrXWZny4l3Y63/bL1D0pYN00KjmfBeH4SKSRt+Js9lDZt0hJ3OzhfqMgt958bYVxsVdL5Hk4m2l6zNMsjh7Qq7lcVXkpm/UxLrKgFz7qkqCKCdK1VeyUaC7mZKH5f6TwKqw7waOxYhdVK3BVhTnT3auGsHW6VHfWADjyzGxg08NaM/28F7Y+yR+hsB+g+bzaMxH7PLicn4shJ1pCfnP79oCjT/F+1Dc4MNPev7vI6qqqWuS7csrbYNVj2uRNR+coGYK2udZw3RrH+tV7g1//oWV5JW6/D+lTA45T9M11OP1UVHU6qL6albMpy9uXatLIrOzZC3TGlJun8W6thuQcKIksn0MHzLFkk14pYqyUW6cpPs1JRm5NoMFXPXzdumCLwG0q/Q6Wx//hW0OqswN2H+SZ5T6/eYrnqmgy30B7/oKGgY0n0JYWj/WNqmdJItMDGmgW6L+PrfXqN+yw==')))
    for name, source in sources.items():
        parts = name.split('.')
        for i in range(1, len(parts)):
            package = '.'.join(parts[:i])
            if package not in sys.modules:
                parent = types.ModuleType(package)
                parent.__path__ = []
                sys.modules[package] = parent
                if i > 1:
                    setattr(sys.modules['.'.join(parts[:i-1])], parts[i-1], parent)
        module = types.ModuleType(name)
        module.__file__ = str(Path.cwd().joinpath(*parts).with_suffix('.py'))
        module.__package__ = '.'.join(parts[:-1])
        sys.modules[name] = module
        exec(compile(source, module.__file__, 'exec'), module.__dict__)
        if len(parts) > 1:
            setattr(sys.modules[module.__package__], parts[-1], module)
_load_notebook_helpers()
del _load_notebook_helpers


In [ ]:
import numpy as np

from pusht import PushTEnv, apply_transform, setup_axes, draw_state
from pusht_diffusion import (
    plt, show_demonstrations, default_policy, show_action_samples, animate,
    OBS_HORIZON, PRED_HORIZON, NUM_DIFFUSION_ITERS,
)


## The environment

**Goal**: push the gray T-block into the green area.  The observation is
`[agent_x, agent_y, block_x, block_y, block_angle]`, and an action is the
position the pusher's PD controller drives toward.

In [ ]:
env = PushTEnv(seed=1000)
obs = env.observation()
action = np.random.uniform(0, 512, size=2)
obs, coverage, done = env.step(action)
with np.printoptions(precision=4, suppress=True):
    print("Obs:   ", obs)
    print("Action:", action)
    print("Coverage of the goal:", coverage)

_, ax = plt.subplots(figsize=(4, 4))
setup_axes(ax)
draw_state(ax, obs)
plt.show()

## The demonstrations

~200 human demonstrations, each a pusher path from a random start until the
block covers the goal.

In [ ]:
show_demonstrations()

## The pretrained policy

The network sees the last `OBS_HORIZON` observations and predicts
`PRED_HORIZON` actions, of which the controller executes `action_horizon`
(8 below) before asking again:

```
|o|o|                             observations: 2
| |a|a|a|a|a|a|a|a|               actions executed: 8
|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p|p| actions predicted: 16
```

In [ ]:
policy = default_policy()
print("device:", policy.device)
print("parameters: %.1fM" % (sum(p.numel() for p in policy.net.parameters()) / 1e6))

## Sampling action sequences

Each sample starts from Gaussian noise and is denoised `NUM_DIFFUSION_ITERS`
times, conditioned on the observation history.  From the same history,
different noise gives different plans.

In [ ]:
# The T at X_WT = (x, y, theta), in the world frame.
# The pusher at p_WPusher = (x, y) in the world frame, but p_TPusher is in the T frame.
# Note that it's using pixel coordinates, with 0,0 in the top left.
X_WT = (219.1, 293.0, np.pi / 4)
p_TPusher = (0, -104)
p_WPusher = apply_transform(X_WT, p_TPusher)
show_action_samples(p_WPusher=p_WPusher, X_WT=X_WT, v_WPusher=(0,0))

## Closed loop

Predict `PRED_HORIZON` actions, execute `action_horizon` of them (starting with the one
for the current time), observe, and predict again.

In [ ]:
# Animate with the pusher at p_WPusher, the T at X_WT.
animate(p_WPusher=p_WPusher, X_WT=X_WT, num_rollouts=4, action_horizon=8)

Now let's run it again, but illustrate multiple samples each time that the diffusion policy inference actually runs.
Red: extra action sequences sampled each time the policy replans, from the same history.
The rollout executes one more sample like them, which isn't drawn.

In [ ]:
animate(num_rollouts=1, max_steps=40, num_extra_samples=8, action_horizon=8)